# Real 2024 Vecchia: GPU-Batched Reverse-L Column

This notebook fits the target-wise GPU-batched reverse-L column Vecchia model.

Implementation:
- kernel: `GEMS_TCO.kernel_vecchia_col_batch.ReverseLColumnVecchiaFitBatch`;
- reverse-L geometry is built on the regular `Latitude/Longitude` grid;
- covariance can use real source coordinates through `keep_ori=True`;
- default conditioning is exactly `14,14,14` for `t, t-1, t-2`, i.e. nominal mature-hour `m=42`;
- `include_lag_self=False` by default, matching the prior column V3 experiments. Set it to `True` if the lag self-anchor should occupy one of the 14 slots.

In [10]:
import os
import sys
import time
import gc
import io
import contextlib
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from torch.nn import Parameter

AMAREL_SRC = '/home/jl2815/tco'
LOCAL_SRC = '/Users/joonwonlee/Documents/GEMS_TCO-1/src'
_src = AMAREL_SRC if os.path.exists(AMAREL_SRC) else LOCAL_SRC
sys.path.insert(0, _src)

from GEMS_TCO import configuration as config
from GEMS_TCO.data_loader import load_data_dynamic_processed
from GEMS_TCO.kernel_vecchia_col_batch import ReverseLColumnVecchiaFitBatch

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
DTYPE = torch.float64

print('SRC:', _src)
print('DEVICE:', DEVICE)
print('torch:', torch.__version__)

ROUND_DECIMALS = 5

def round_numeric_df(df, digits=ROUND_DECIMALS):
    out = df.copy()
    num_cols = out.select_dtypes(include=[np.number]).columns
    out[num_cols] = out[num_cols].round(digits)
    return out

def round_numeric_series(s, digits=ROUND_DECIMALS):
    out = s.copy()
    for idx, val in out.items():
        if isinstance(val, (float, np.floating)) and np.isfinite(val):
            out[idx] = round(float(val), digits)
    return out


SRC: /Users/joonwonlee/Documents/GEMS_TCO-1/src
DEVICE: cpu
torch: 2.5.1


## Settings

In [11]:
YEAR = '2024'
MONTH = 7
DAYS_LIST = [13]  # 0-based: [13] is July 14. Full July: list(range(31))
LAT_RANGE = [-3, 2]
LON_RANGE = [121, 131]
LAT_LON_RESOLUTION = [1, 1]

SMOOTH = 0.5
USE_SOURCE_COORDS_FOR_COVARIANCE = True  # keep_ori=True; grid_coords still define reverse-L scan

# Reverse-L column V3.  per_lag_conditioning_count is the cap for each of t, t-1, t-2.
HEAD_RIGHT_COLS = 0
ABOVE_COUNT = 2
RIGHT_COL_COUNT = 3
PER_LAG_CONDITIONING_COUNT = 14
LAG_COUNT = 2
INCLUDE_LAG_SELF = False
TARGET_CHUNK_SIZE = 1024  # lower to 512 if GPU memory is tight

TOTAL_CONDITIONING_NOMINAL = PER_LAG_CONDITIONING_COUNT * (LAG_COUNT + 1)

LBFGS_LR = 1.0
LBFGS_MAX_STEPS = 20
LBFGS_MAX_EVAL = 20
LBFGS_HISTORY_SIZE = 10
GRAD_TOL = 1e-5
SUPPRESS_FIT_PRINTS = False

OUT_DIR = Path('/Users/joonwonlee/Documents/GEMS_TCO-1/Exercises/st_model/day/local_computer/log')
OUT_PREFIX = 'real_vecc_2024_column_batched_051326'
SAVE_CSV = True

INIT_PHYSICAL = {
    'sigmasq': 10.0,
    'range_lat': 0.30,
    'range_lon': 0.40,
    'range_time': 2.0,
    'advec_lat': 0.05,
    'advec_lon': -0.10,
    'nugget': 2.5,
}

print('days:', DAYS_LIST)
print('smooth:', SMOOTH)
print('coord mode:', 'source' if USE_SOURCE_COORDS_FOR_COVARIANCE else 'grid')
print('reverse-L:', {
    'head_right_cols': HEAD_RIGHT_COLS,
    'above_count': ABOVE_COUNT,
    'right_col_count': RIGHT_COL_COUNT,
    'per_lag_conditioning_count': PER_LAG_CONDITIONING_COUNT,
    'lag_count': LAG_COUNT,
    'include_lag_self': INCLUDE_LAG_SELF,
    'total_conditioning_nominal': TOTAL_CONDITIONING_NOMINAL,
})

days: [13]
smooth: 0.5
coord mode: source
reverse-L: {'head_right_cols': 0, 'above_count': 2, 'right_col_count': 3, 'per_lag_conditioning_count': 14, 'lag_count': 2, 'include_lag_self': False, 'total_conditioning_nominal': 42}


## Load Real Data

The column kernel does not need point max-min ordering or `nns_map`. It uses the grid coordinates to build the reverse-L scan and skips missing target/neighbor values during precompute.

In [12]:
data_load_instance = load_data_dynamic_processed(config.mac_data_load_path)

df_map, _, _, monthly_mean = data_load_instance.load_maxmin_ordered_data_bymonthyear(
    lat_lon_resolution=LAT_LON_RESOLUTION,
    mm_cond_number=1,
    years_=[YEAR],
    months_=[MONTH],
    lat_range=LAT_RANGE,
    lon_range=LON_RANGE,
    is_whittle=True,
)

key_idx = sorted(df_map)
print('n hourly slots:', len(key_idx))
print('monthly_mean:', monthly_mean)
print('first key:', key_idx[0], 'last key:', key_idx[-1])

base_grid_coords_np = df_map[key_idx[0]][['Latitude', 'Longitude']].to_numpy(dtype=np.float64)
print('base_grid_coords_np:', base_grid_coords_np.shape)
print('n unique lat/lon:', len(np.unique(base_grid_coords_np[:,0])), len(np.unique(base_grid_coords_np[:,1])))

--- Global Monthly Mean for 2024-7: 257.9726 ---
n hourly slots: 248
monthly_mean: 257.9726104252314
first key: 2024_07_y24m07day01_hm00:53 last key: 2024_07_y24m07day31_hm07:48
base_grid_coords_np: (18126, 2)
n unique lat/lon: 114 159


In [13]:
def assert_grid_order_consistent(keys, base_coords):
    for k in keys:
        coords = df_map[k][['Latitude', 'Longitude']].to_numpy(dtype=np.float64)
        if coords.shape != base_coords.shape or not np.allclose(coords, base_coords, equal_nan=True):
            raise RuntimeError(f'Grid coordinate order differs at {k}; reverse-L local-index mapping is not reusable.')

for day_idx in DAYS_LIST:
    hour_indices = [day_idx * 8, (day_idx + 1) * 8]
    assert_grid_order_consistent(key_idx[hour_indices[0]:hour_indices[1]], base_grid_coords_np)
print('grid order consistency check passed for selected days')

grid order consistency check passed for selected days


In [14]:
daily_hourly_maps = {}
selected_key_map = {}

for day_idx in DAYS_LIST:
    hour_indices = [day_idx * 8, (day_idx + 1) * 8]
    selected_key_map[day_idx] = key_idx[hour_indices[0]:hour_indices[1]]
    day_hourly_map, _ = data_load_instance.load_working_data(
        df_map,
        monthly_mean,
        hour_indices,
        ord_mm=None,
        dtype=DTYPE,
        keep_ori=USE_SOURCE_COORDS_FOR_COVARIANCE,
    )
    daily_hourly_maps[day_idx] = {k: v.to(DEVICE) for k, v in day_hourly_map.items()}

rows = []
for day_idx, maps in daily_hourly_maps.items():
    n_valid = sum(int((~torch.isnan(v[:, 2])).sum().item()) for v in maps.values())
    n_total = sum(int(v.shape[0]) for v in maps.values())
    rows.append({
        'day_idx': day_idx,
        'day': f'{YEAR}-{MONTH:02d}-{day_idx + 1:02d}',
        'n_time_slots': len(maps),
        'n_rows_total': n_total,
        'n_valid_o3': n_valid,
        'valid_rate': n_valid / max(n_total, 1),
        'first_slot': selected_key_map[day_idx][0] if selected_key_map[day_idx] else None,
        'last_slot': selected_key_map[day_idx][-1] if selected_key_map[day_idx] else None,
    })
load_summary = pd.DataFrame(rows)
display(round_numeric_df(load_summary))

,day_idx,day,n_time_slots,n_rows_total,n_valid_o3,valid_rate,first_slot,last_slot
0,13,2024-07-14,8,145008,144078,0.9936,2024_07_y24m07day14_hm00:53,2024_07_y24m07day14_hm07:48


## Parameter Helpers

In [15]:
P_LABELS = ['sigmasq', 'range_lat', 'range_lon', 'range_time', 'advec_lat', 'advec_lon', 'nugget']

def physical_to_log_phi(params):
    sigmasq = float(params['sigmasq'])
    range_lat = float(params['range_lat'])
    range_lon = float(params['range_lon'])
    range_time = float(params['range_time'])
    nugget = float(params['nugget'])
    phi2 = 1.0 / range_lon
    phi1 = sigmasq * phi2
    phi3 = (range_lon / range_lat) ** 2
    phi4 = (range_lon / range_time) ** 2
    return [
        np.log(phi1), np.log(phi2), np.log(phi3), np.log(phi4),
        float(params['advec_lat']), float(params['advec_lon']), np.log(nugget),
    ]

def backmap_params(out_params):
    p = [x.item() if isinstance(x, torch.Tensor) else float(x) for x in out_params[:7]]
    phi2 = np.exp(p[1])
    phi3 = np.exp(p[2])
    phi4 = np.exp(p[3])
    rlon = 1.0 / phi2
    return {
        'sigmasq': np.exp(p[0]) / phi2,
        'range_lat': rlon / phi3 ** 0.5,
        'range_lon': rlon,
        'range_time': rlon / phi4 ** 0.5,
        'advec_lat': p[4],
        'advec_lon': p[5],
        'nugget': np.exp(p[6]),
    }

def make_params_list(init_physical=INIT_PHYSICAL):
    initial_vals = physical_to_log_phi(init_physical)
    return [Parameter(torch.tensor([val], dtype=DTYPE, device=DEVICE)) for val in initial_vals]

print('init log phi:', np.round(physical_to_log_phi(INIT_PHYSICAL), 4))

init log phi: [ 3.2189  0.9163  0.5754 -3.2189  0.05   -0.1     0.9163]


## Fit Column-Batched Reverse-L

In [16]:
def instantiate_model(day_map):
    return ReverseLColumnVecchiaFitBatch(
        smooth=SMOOTH,
        input_map=day_map,
        mm_cond_number=0,
        nheads=0,
        grid_coords=base_grid_coords_np,
        head_right_cols=HEAD_RIGHT_COLS,
        above_count=ABOVE_COUNT,
        right_col_count=RIGHT_COL_COUNT,
        per_lag_conditioning_count=PER_LAG_CONDITIONING_COUNT,
        lag_count=LAG_COUNT,
        include_lag_self=INCLUDE_LAG_SELF,
        target_chunk_size=TARGET_CHUNK_SIZE,
        use_data_coords_for_offsets=USE_SOURCE_COORDS_FOR_COVARIANCE,
    )

def batch_diagnostics(model):
    rows = []
    for g in model.Batched_Groups:
        real_m = (~g['is_dummy']).sum(dim=1).detach().cpu().numpy()
        rows.append({
            'max_m': int(g['max_m']),
            'n_targets': int(g['target_idx'].shape[0]),
            'mean_m': float(np.mean(real_m)) if real_m.size else np.nan,
            'median_m': float(np.median(real_m)) if real_m.size else np.nan,
            'max_real_m': int(np.max(real_m)) if real_m.size else 0,
        })
    return rows

def fit_one(day_idx):
    day_map = daily_hourly_maps[day_idx]
    params_list = make_params_list()
    model = instantiate_model(day_map)

    t0 = time.time()
    if SUPPRESS_FIT_PRINTS:
        with contextlib.redirect_stdout(io.StringIO()):
            model.precompute_conditioning_sets()
    else:
        model.precompute_conditioning_sets()
    precompute_s = time.time() - t0

    optimizer = model.set_optimizer(
        params_list,
        lr=LBFGS_LR,
        max_iter=LBFGS_MAX_EVAL,
        max_eval=LBFGS_MAX_EVAL,
        history_size=LBFGS_HISTORY_SIZE,
    )

    t1 = time.time()
    if SUPPRESS_FIT_PRINTS:
        with contextlib.redirect_stdout(io.StringIO()):
            out, steps_ran = model.fit_vecc_lbfgs(params_list, optimizer, max_steps=LBFGS_MAX_STEPS, grad_tol=GRAD_TOL)
    else:
        out, steps_ran = model.fit_vecc_lbfgs(params_list, optimizer, max_steps=LBFGS_MAX_STEPS, grad_tol=GRAD_TOL)
    fit_s = time.time() - t1

    est = backmap_params(out)
    bdiag = batch_diagnostics(model)
    valid_count = sum(int((~torch.isnan(v[:, 2])).sum().item()) for v in day_map.values())
    row = {
        'year': YEAR,
        'month': MONTH,
        'day_idx': int(day_idx),
        'day': f'{YEAR}-{MONTH:02d}-{day_idx + 1:02d}',
        'model': 'ColumnV3Batched_ReverseL_14_14_14',
        'kernel': 'column_reverse_l_v3_batched',
        'coord_mode': 'Source_Latitude/Source_Longitude' if USE_SOURCE_COORDS_FOR_COVARIANCE else 'Latitude/Longitude grid',
        'smooth': SMOOTH,
        'loss': float(out[-1]),
        'fit_steps_raw': int(steps_ran),
        'precompute_s': round(precompute_s, ROUND_DECIMALS),
        'fit_s': round(fit_s, ROUND_DECIMALS),
        'total_s': round(precompute_s + fit_s, ROUND_DECIMALS),
        'n_valid_o3': int(valid_count),
        'n_heads': int(model.Heads_data.shape[0]),
        'n_tails': int(model.n_tails),
        'n_batches': len(model.Batched_Groups),
        'total_conditioning_nominal': TOTAL_CONDITIONING_NOMINAL,
        'head_right_cols': HEAD_RIGHT_COLS,
        'above_count': ABOVE_COUNT,
        'right_col_count': RIGHT_COL_COUNT,
        'per_lag_conditioning_count': PER_LAG_CONDITIONING_COUNT,
        'lag_count': LAG_COUNT,
        'include_lag_self': INCLUDE_LAG_SELF,
        'target_chunk_size': TARGET_CHUNK_SIZE,
        **{f'est_{k}': float(est[k]) for k in P_LABELS},
    }
    if bdiag:
        diag_df = pd.DataFrame(bdiag)
        row.update({
            'largest_batch_n': int(diag_df['n_targets'].max()),
            'mean_m_over_batches': float(np.average(diag_df['mean_m'], weights=diag_df['n_targets'])),
            'median_m_over_batches': float(np.median(np.repeat(diag_df['median_m'].to_numpy(), diag_df['n_targets'].to_numpy()))),
            'max_real_m': int(diag_df['max_real_m'].max()),
        })
    del model, params_list, optimizer
    gc.collect()
    if DEVICE.type == 'cuda':
        torch.cuda.empty_cache()
    return row

records = []
for day_idx in DAYS_LIST:
    print(f'\n=== fitting day_idx={day_idx} ({YEAR}-{MONTH:02d}-{day_idx+1:02d}) ===')
    try:
        row = fit_one(day_idx)
        records.append(row)
        print(round_numeric_series(pd.Series(row)).to_string())
    except Exception as exc:
        import traceback
        print(f'FAILED: {type(exc).__name__}: {exc}')
        traceback.print_exc()
        records.append({
            'year': YEAR, 'month': MONTH, 'day_idx': int(day_idx),
            'day': f'{YEAR}-{MONTH:02d}-{day_idx + 1:02d}',
            'model': 'ColumnV3Batched_ReverseL_14_14_14',
            'kernel': 'column_reverse_l_v3_batched',
            'error': repr(exc),
        })

results_df = pd.DataFrame(records)
display(round_numeric_df(results_df))


=== fitting day_idx=13 (2024-07-14) ===
Pre-computing Batched ReverseLColumn V3 [heads_right=0, above=2, right_cols=3, per_lag=14, lags=2, coord_mode=data]... Done in 1.6s. grid=114x159, heads=0, tails=144078, batches=[(14, 18068), (28, 18069), (42, 107941)], m mean/med/max=36.1/42/42
--- Starting Batched Reverse-L Vecchia L-BFGS ---
--- Step 1/20 / Loss: 1.1145 / Max Grad: 2.38e-05 ---
--- Step 2/20 / Loss: 1.0723 / Max Grad: 6.10e-06 ---
Converged: max_grad 6.10e-06 < 1.00e-05
Final Interpretable Params: {'sigma_sq': 7.0185, 'range_lon': 0.2441, 'range_lat': 0.196, 'range_time': 1.123, 'advec_lat': -0.0477, 'advec_lon': -0.21, 'nugget': 1.0607}
year                                                       2024
month                                                         7
day_idx                                                      13
day                                                  2024-07-14
model                         ColumnV3Batched_ReverseL_14_14_14
kernel                  

,year,month,day_idx,day,model,kernel,coord_mode,smooth,loss,fit_steps_raw,...,est_range_lat,est_range_lon,est_range_time,est_advec_lat,est_advec_lon,est_nugget,largest_batch_n,mean_m_over_batches,median_m_over_batches,max_real_m
0,2024,7,13,2024-07-14,ColumnV3Batched_ReverseL_14_14_14,column_reverse_l_v3_batched,Source_Latitude/Source_Longitude,0.5,1.0723,1,...,0.196,0.2441,1.123,-0.0477,-0.21,1.0607,107941,36.1437,42.0,42


In [17]:
if SAVE_CSV:
    OUT_DIR.mkdir(parents=True, exist_ok=True)
    coord_tag = 'source' if USE_SOURCE_COORDS_FOR_COVARIANCE else 'grid'
    out_path = OUT_DIR / f'{OUT_PREFIX}_{coord_tag}_days_{min(DAYS_LIST)+1:02d}_{max(DAYS_LIST)+1:02d}.csv'
    round_numeric_df(results_df).to_csv(out_path, index=False, float_format="%.5f")
    print('saved:', out_path)

saved: /Users/joonwonlee/Documents/GEMS_TCO-1/Exercises/st_model/day/local_computer/log/real_vecc_2024_column_batched_051326_source_days_14_14.csv
